## fine-tuning

Before starting, set the "runtime" to a T4 GPU (click on the little down arrow on the right side, next to "RAM Disk" icon)

In [1]:
import torch
from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
)
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

/Users/fcalado/Desktop/anti-trans-legislation/training/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Add this to monitor MPS memory usage
def print_mps_memory():
    if torch.backends.mps.is_available():
        print(f"MPS allocated: {torch.mps.current_allocated_memory() / 1024**3:.2f} GB")
        print(f"MPS cached: {torch.mps.driver_allocated_memory() / 1024**3:.2f} GB")

# Call this periodically during training
print_mps_memory()

MPS allocated: 0.00 GB
MPS cached: 0.00 GB


In [3]:
# Check if MPS is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found.")
else:
    device = torch.device("cpu")
    print("MPS device not found, using CPU.")

MPS device found.


In [4]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
model = model.to(device)  # Move model to MPS

In [5]:
ds = load_dataset("gofilipa/aclu_transgender")

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 6875
    })
})

In [ ]:
# Set environment variables for better performance
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [ ]:
# Clear memory first
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Training configuration with proper parameters
training_params = SFTConfig(
    output_dir="../checkpoints",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=2, # moving slowly from 1 to 3
    learning_rate=2e-4,
    weight_decay=0.001,
    dataset_text_field="text",
    report_to="none",
    bf16=False,
    fp16=False,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    max_seq_length=512,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    save_strategy="epoch",
    logging_steps=10,
    average_tokens_across_devices=False  # Fix for single device training
    # Remove loss_type parameter to avoid the warning
    # The trainer will automatically use ForCausalLMLoss which is correct
)

# Configure model for gradient checkpointing compatibility
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    processing_class=tokenizer,
    args=training_params
)

In [10]:
# Add this to monitor MPS memory usage
def print_mps_memory():
    if torch.backends.mps.is_available():
        print(f"MPS allocated: {torch.mps.current_allocated_memory() / 1024**3:.2f} GB")
        print(f"MPS cached: {torch.mps.driver_allocated_memory() / 1024**3:.2f} GB")

# Call this periodically during training
print_mps_memory()

MPS allocated: 0.48 GB
MPS cached: 1.02 GB


In [11]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.000200
20,3.823600
30,3.694900
40,3.851000
50,3.668000
60,3.749700
70,3.599300
80,3.515400
90,3.706600
100,3.594400


RuntimeError: MPS backend out of memory (MPS allocated: 3.09 GB, other allocations: 14.91 GB, max allowed: 18.13 GB). Tried to allocate 147.24 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [18]:
trainer.model.save_pretrained("../models/gpt2-aclu-e3")
trainer.tokenizer.save_pretrained("../models/gpt2-aclu-e3")

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


('../models/gpt2-aclu-e3/tokenizer_config.json',
 '../models/gpt2-aclu-e3/special_tokens_map.json',
 '../models/gpt2-aclu-e3/vocab.json',
 '../models/gpt2-aclu-e3/merges.txt',
 '../models/gpt2-aclu-e3/added_tokens.json',
 '../models/gpt2-aclu-e3/tokenizer.json')

In [19]:
model = AutoModelForCausalLM.from_pretrained("../models/gpt2-aclu-e3")
tokenizer = AutoTokenizer.from_pretrained("../models/gpt2-aclu-e3")

In [20]:
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_length=50)

Device set to use mps:0


In [21]:
pipe("Gender identity is defined as")

[{'generated_text': 'Gender identity is defined as the expression of a desire to change society — changing gender markers, changing clothes, changing hair etc.'}]

In [22]:
pipe("The right of")

[{'generated_text': 'The right of self-determination and self-expression to be a part of the fabric of our nation is deeply rooted.'}]

In [23]:
pipe("Transgender")

[{'generated_text': 'Transgender youth now face an ever-changing body — even more complicated and more difficult to navigate.'}]

In [24]:
pipe("Transgender students")

[{'generated_text': 'Transgender students and teachers are supposed to be respectful of transgender students and teachers.'}]

In [26]:
# Delete specific large variables
del model, tokenizer, trainer
del ds  # Delete the dataset

# Clear all variables (use with caution)
# %reset -f

# Force garbage collection
import gc
gc.collect()

# Clear PyTorch cache if using GPU/MPS
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

NameError: name 'model' is not defined